# YOLO11n License Plate Detection and OCR

Run the cells from top to bottom in Google Colab with a GPU runtime selected. When prompted, upload exactly these three files: the updated `detect_plate.py`, `plate_detector_yolo11n.pt`, and `video.mp4`. The notebook runs detection and OCR, displays a sample annotated frame and plate summary, and downloads all results as a ZIP.

In [ ]:
%pip install -q ultralytics "fast-plate-ocr[onnx]" opencv-python-headless scipy

In [ ]:
import os
import platform
import torch
from google.colab import files

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Select Runtime > Change runtime type > T4 GPU, then rerun this cell.")

os.chdir("/content")
os.environ["ANPR_VIDEO_PATH"] = "/content/video.mp4"
os.environ["ANPR_SHOW_PREVIEW"] = "0"
required_files = {"detect_plate.py", "plate_detector_yolo11n.pt", "video.mp4"}
missing_files = {name for name in required_files if not os.path.isfile(name)}
if missing_files:
    print("Choose these files in the upload dialog:", sorted(missing_files))
    files.upload()
    missing_files = {name for name in required_files if not os.path.isfile(name)}
if missing_files:
    raise FileNotFoundError(f"Upload was incomplete. Missing in /content: {sorted(missing_files)}")
print("All three inputs are ready in /content.")

In [ ]:
import os
import runpy
import zipfile
import cv2
import pandas as pd
from PIL import Image
from google.colab import files
from IPython.display import display

runpy.run_path("/content/detect_plate.py", run_name="__main__")

output_video = "/content/output_annotated.mp4"
summary_csv = "/content/plate_summary.csv"
if not os.path.isfile(output_video):
    raise FileNotFoundError("Inference did not create output_annotated.mp4. Check the pipeline messages above.")

if os.path.isfile(summary_csv):
    print("Recognized plates:")
    display(pd.read_csv(summary_csv))

capture = cv2.VideoCapture(output_video)
frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
if capture.isOpened() and frame_count > 0:
    capture.set(cv2.CAP_PROP_POS_FRAMES, frame_count // 2)
    ok, preview_frame = capture.read()
    if ok:
        display(Image.fromarray(cv2.cvtColor(preview_frame, cv2.COLOR_BGR2RGB)))
capture.release()

result_files = [
    "output_annotated.mp4",
    "plate_log.csv",
    "plate_raw_reads.csv",
    "plate_summary.csv",
]
with zipfile.ZipFile("/content/anpr_results.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in result_files:
        if os.path.isfile(path):
            archive.write(path, arcname=os.path.basename(path))
    if os.path.isdir("/content/cropped_plates"):
        for root, _, filenames in os.walk("/content/cropped_plates"):
            for filename in filenames:
                path = os.path.join(root, filename)
                archive.write(path, arcname=os.path.relpath(path, "/content"))

print("Created anpr_results.zip with video, CSV logs, and plate crops.")
files.download("/content/anpr_results.zip")